In [1]:
import os
import csv
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from pathlib import Path
import pandas as pd
import sys
from tqdm.auto import tqdm
from scipy.spatial.distance import euclidean
from dtaidistance import dtw
from tqdm.auto import tqdm

project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.seed import set_seed
from src.data.preprocessing.pipeline import Pipeline
from src.data.datasets.universal_dataset import CVADataset
from src.transformer.network import NARCVGenerator 
from src.train.trainer_transformer import CSTrainer
from scipy.signal import savgol_filter

/mnt/c/Users/edtop/ITMO/THESIS_CV_DIFF/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
EPOCHS = 30
BATCH_SIZE = 64
LR = 0.0004
WEIGHT_DECAY = 0.025

NUM_CYCLE = [1, 2, 3, 4]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEQ_LEN = 968         
PATCH_SIZE = 8
D_MODEL = 64
N_HEAD = 4
D_INNER = 32
NUM_LAYERS = 4
NUM_COND_TOKENS = 4
DROPOUT = 0.2

In [ ]:
def compute_dtw(original, reconstructed):
    orig_f64 = np.array(original, dtype=np.float64).flatten()
    recon_f64 = np.array(reconstructed, dtype=np.float64).flatten()
    from dtaidistance import dtw
    return dtw.distance_fast(orig_f64, recon_f64, use_c=True)

test_inhibitor = "2-mercaptobenzimidazole" 

NEW_RESULT_VARIABLE = "_transformer"
RESULTS_CSV = os.path.join(project_root, "experiments", f"loio_results{NEW_RESULT_VARIABLE}_CYCLE_INFORMED.csv")

print("\n" + "="*60)
print(f"LOIO ТЕСТ (Transformer) | Оставляем для валидации: {test_inhibitor}")
print("="*60)

set_seed(42)

save_dir = os.path.join(project_root, "experiments", f"run_NAR_cv_{test_inhibitor}")
os.makedirs(save_dir, exist_ok=True)

pipe = Pipeline(
    num_cycle=NUM_CYCLE, 
    test_inhibitor=test_inhibitor, 
    norm_feat=True, 
    use_wavelet=False, 
    flip_the_peak=False
)

train_dataset = CVADataset(vol=pipe.train_voltage, cur=pipe.train_current, desc_df=pipe.train_analyzed_data)
val_dataset = CVADataset(vol=pipe.test_voltage, cur=pipe.test_current, desc_df=pipe.test_analyzed_data)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

num_desc_features = train_dataset[0]["features"].shape[0]
SEQ_LEN = train_dataset[0]["current"].shape[-1]

print("\n" + "-"*40)
print("[DEBUG] ПРОВЕРКА ДАННЫХ ПЕРЕД ОБУЧЕНИЕМ")
sample_batch = next(iter(train_loader))
print(f"Размерность признаков химии: {sample_batch['features'].shape} -> Ожидаем [64, {num_desc_features}]")
print(f"Размерность тензора цикла:   {sample_batch['cycle_num'].shape} -> Ожидаем [64, 1]")
print(f"Первые 5 нормализованных циклов в батче:\n{sample_batch['cycle_num'][:5].squeeze().numpy()}")
print("-"*40 + "\n")

model = NARCVGenerator(
    desc_dim=num_desc_features,
    seq_len=SEQ_LEN,
    patch_size=PATCH_SIZE,
    d_model=D_MODEL,
    n_head=N_HEAD,
    d_inner=D_INNER,
    num_layers=NUM_LAYERS,
    num_cond_tokens=NUM_COND_TOKENS,
    dropout=DROPOUT,
    use_cycle_feat=True
).to(DEVICE)

trainer = CSTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    save_dir=str(save_dir),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    epochs=EPOCHS,
    use_smoothing=True
)

print(f"[*] Обучение Трансформера без {test_inhibitor}...")
trainer.fit()

print(f"[*] Запуск Инференса и расчет метрик для {test_inhibitor}...") 

model.load_state_dict(torch.load(os.path.join(save_dir, "best_model.pth"), map_location=DEVICE)['model_state_dict'])
model.eval()

dtw_scores = []
mae_scores = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc=f"Eval {test_inhibitor}"):
        current = batch["current"].to(DEVICE)
        features = batch["features"].to(DEVICE)
        cycle_num = batch["cycle_num"].to(DEVICE)
        
        shape_pred, _ = model(features, cycle_num)

        true_np = current.squeeze(1).cpu().numpy()     
        pred_np = shape_pred.squeeze(1).cpu().numpy()
        
        pred_np_smoothed = savgol_filter(
            pred_np, 
            window_length=27, 
            polyorder=3, 
            axis=-1
        )
        for j in range(true_np.shape[0]):
            orig_signal = true_np[j]
            gen_signal = pred_np_smoothed[j]
            
            mae = np.mean(np.abs(orig_signal - gen_signal))
            dtw_dist = compute_dtw(orig_signal, gen_signal)
            
            mae_scores.append(mae)
            dtw_scores.append(dtw_dist)

final_dtw = np.mean(dtw_scores)
final_mae = np.mean(mae_scores)

print(f"\nРезультат для {test_inhibitor}: DTW = {final_dtw:.4f} | MAE = {final_mae:.6f}")

file_exists = os.path.isfile(RESULTS_CSV)

with open(RESULTS_CSV, mode='a', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(['Inhibitor', 'DTW', 'MAE'])
    
    writer.writerow([test_inhibitor, final_dtw, final_mae])

print(f"[*] Данные успешно сохранены в {RESULTS_CSV}")


LOIO ТЕСТ (Transformer) | Оставляем для валидации: 2-mercaptobenzimidazole

----------------------------------------
[DEBUG] ПРОВЕРКА ДАННЫХ ПЕРЕД ОБУЧЕНИЕМ
Размерность признаков химии: torch.Size([64, 41]) -> Ожидаем [64, 41]
Размерность тензора цикла:   torch.Size([64, 1]) -> Ожидаем [64, 1]
Первые 5 нормализованных циклов в батче:
[0.         0.         0.33333334 0.         0.6666667 ]
----------------------------------------



Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.2+cu121
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


[*] Обучение Трансформера без 2-mercaptobenzimidazole...
Training NAR Transformer on cuda...


Epoch 1 [Val]: 100%|██████████| 13/13 [00:03<00:00,  4.31it/s, val_loss=1.3322]
/mnt/c/Users/edtop/ITMO/THESIS_CV_DIFF/src/utils/plots2.py:60: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(save_path, dpi=150, bbox_inches='tight')
